# SHAPE — Self-Healing Autonomous Programming Engine
## Notebook 1 of 4: System Architecture & Agent Initialization

**Group 3 | ITAI2376-14562**  
Cherluk Sumdin · Olubukola Kukoyi · Taki Eddine Boubekri · Michael Rivas

---

This notebook walks through SHAPE's three-tier hardware architecture, explains every
design decision in the codebase, and runs the agent initialization from source so
every component can be verified live. This is **not** a demo script — it is a live
engineering walkthrough of the code we built.


## 1. What SHAPE Is (and Is Not)

SHAPE is a **prompt-based ReAct agent** running fully offline on consumer hardware.
The key architectural constraint: **no native tool-calling API** is available because
the underlying model (CodeLlama 7B via Ollama) only exposes a text-completion
interface. We therefore implemented our own ReAct loop and parser on top of
`ChatOllama` — this is the central technical contribution of `shape/agent.py`.

Unlike typical course demos, SHAPE runs across **three physical devices**:

| Tier | Hardware | Software | Role |
|---|---|---|---|
| Tier 1 | Raspberry Pi Pico 2 (RP2350) | MicroPython | Sensor firmware, ST7789 LCD display |
| Tier 2 | Raspberry Pi Zero 2 W | Linux + Python | Field terminal: HAT display, keys, voice |
| Tier 3 | Raspberry Pi 5 (8 GB) | Linux + Ollama | Full SHAPE agent + HTTP API |

Data flow:
```
[Pico 2]  --UART 115200--> [Zero 2 W] --HTTP Wi-Fi--> [Pi 5]
 sensors                   field terminal             SHAPE agent
 ST7789 LCD                ST7789 HAT                 Ollama LLM
 sensors.py/main.py        serial_bridge.py           api_server.py
```


## 2. Tier-1 → Tier-2 UART Protocol (hardware/pico2/main.py)

The Pico 2 firmware sends one CSV telemetry line every 2 seconds over UART (GPIO 0/1):
```
SHAPE,{ts_ms},{temp_c:.2f},{humidity_pct:.1f},{aqi:.0f},{alert:d}\r\n
```
Example: `SHAPE,14520,22.40,58.1,312,0`

Return path (Pi 5 → Zero 2 W → Pico 2 LCD):
```json
{"cmd": "STATUS", "text": "Sensors nominal", "reward": 0.85}
{"cmd": "ALERT",  "text": "AQI=623 — open windows"}
```
The `reward` field (RL avg score normalized 0-1) renders as a coloured progress bar
on the Pico 2 screen via `draw_status_screen()` in `hardware/pico2/main.py`.


In [ ]:
# Reproduce the exact UART CSV format sent by the Pico 2 firmware.
# This mirrors _send_csv() in hardware/pico2/main.py (MicroPython).

import random
random.seed(99)

def pico_csv_line(ts_ms, temp, humidity, aqi, alert):
    """Mirrors _send_csv() in hardware/pico2/main.py."""
    return f"SHAPE,{ts_ms},{temp:.2f},{humidity:.1f},{aqi:.0f},{int(alert)}"

print(f"{'Line':<5} {'Raw CSV (115200 baud UART)':<50} {'flag'}")
print("-" * 72)
for i in range(10):
    ts    = i * 2000
    t     = round(random.gauss(23, 2), 2)
    h     = round(random.gauss(55, 6), 1)
    aqi   = round(random.gauss(320, 90))
    # Alert threshold: same as api_server.py /sensor handler
    alert = aqi > 600 or t > 40
    line  = pico_csv_line(ts, t, h, aqi, alert)
    flag  = "  <-- ALERT -> auto-query" if alert else ""
    print(f"  {i+1:<3} {line:<50}{flag}")

print()
print("Return-path JSON (Pi 5 -> Zero 2 W -> Pico 2 LCD):")
import json
for cmd in [
    {"cmd": "STATUS", "text": "Sensors nominal", "reward": 0.85},
    {"cmd": "ALERT",  "text": "AQI=623 — open windows"},
    {"cmd": "RESET"},
]:
    print(f"  {json.dumps(cmd)}")

## 3. The ReAct Agent Loop (shape/agent.py)

Because `ChatOllama` has no tool-calling API, our `SHAPEAgent.run()` method
implements the full ReAct cycle manually:

```
SystemMessage (tool descriptions + optional RL policy hint)
   │
   ▼  for step in range(max_iterations=8):
llm.invoke(messages)                 ← one forward pass through Ollama
   │
   ▼
_parse_llm_output(text)
   ├── 'final'  → break, return answer
   └── 'action' → tool_name + tool_input
                      │
                      ▼
                 tools_by_name[tool_name].invoke(tool_input)
                      │
                      ▼
                 observation appended as HumanMessage('Observation: ...')
                      │
                      └─────────────────────── next iteration
```

The RL policy hint (from `FeedbackMemory.get_policy_hint()`) is injected into
the system prompt at the **start of every** `run()` call, so the agent's
behaviour adapts as ratings accumulate across sessions.


In [ ]:
# Demonstrate _parse_llm_output() — the exact parser from shape/agent.py.
# Tests all three branches: Action, Final Answer, and plain-text fallback.

import re

def parse_llm_output(text: str) -> dict:
    """Verbatim copy of SHAPEAgent._parse_llm_output() from shape/agent.py."""
    fa = re.search(r"Final Answer\s*:\s*(.+)", text, re.DOTALL | re.IGNORECASE)
    if fa:
        return {"type": "final", "answer": fa.group(1).strip()}
    act   = re.search(r"Action\s*:\s*(.+)", text)
    inp   = re.search(r"Action Input\s*:\s*(.+)", text, re.DOTALL)
    if act:
        return {"type": "action", "tool": act.group(1).strip(),
                "input": inp.group(1).strip() if inp else ""}
    return {"type": "final", "answer": text.strip()}   # graceful fallback

cases = [
    ("Thought: Lint first.\nAction: linter\nAction Input: def f(): pass",
     "Branch 1 — Action"),
    ("Thought: Done.\nFinal Answer: def fib(n): return n if n<2 else fib(n-1)+fib(n-2)",
     "Branch 2 — Final Answer"),
    ("Here is the solution: print('hello')",
     "Branch 3 — Plain text fallback (no tags)"),
]

for text, label in cases:
    r = parse_llm_output(text)
    print(f"\n{label}")
    print(f"  Input: {repr(text[:60])}")
    print(f"  → type={r['type']}", end="")
    if r['type'] == 'action':
        print(f", tool={r['tool']!r}, input={repr(r['input'][:40])}")
    else:
        print(f", answer={repr(r['answer'][:50])}")

In [ ]:
# Inspect the tool registry from shape/tools/__init__.py.
# Shows that dispatch is O(1) via tools_by_name dict,
# and that unknown tool names return a recoverable error Observation.

import sys, os
sys.path.insert(0, os.path.abspath(".."))

from shape.tools import get_tools

config = {
    "executor_timeout": 10, "executor_max_output": 4096,
    "workspace_dir": "../workspace",
    "doc_index_path": "../data/doc_index",
    "docs_dir": "../data/docs",
    "doc_top_k": 3,
}

tools          = get_tools(config)
tools_by_name  = {t.name: t for t in tools}

print(f"Registered tools ({len(tools)}):")
print(f"  {'Name':<20}  Description (first sentence)")
print("-" * 75)
for t in tools:
    print(f"  {t.name:<20}  {t.description.split('.')[0].strip()}.")

print()
bad = "web_browser"
recovered = f"Error: unknown tool '{bad}'. Available: {list(tools_by_name)}"
print("Hallucinated-tool recovery (from agent.py:run):")
print(f"  LLM requested tool: {bad!r}")
print(f"  Observation returned: {recovered}")

In [ ]:
# Initialize the full SHAPE agent — same path as main.py on the Pi 5.
# verbose=True makes the ReAct trace visible in notebook output cells.

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(name)s — %(message)s")

from shape.agent import build_agent

agent_config = {
    "model":            "codellama",
    "ollama_url":       "http://localhost:11434",
    "temperature":      0.2,
    "context_window":   4096,
    "memory_window":    6,
    "max_iterations":   8,
    "executor_timeout": 10,
    "feedback_path":    "../data/feedback.jsonl",
    "workspace_dir":    "../workspace",
    "doc_index_path":   "../data/doc_index",
    "docs_dir":         "../data/docs",
    "doc_top_k":        3,
    "verbose":          True,
}

agent = build_agent(agent_config)

print("\nAgent component report:")
print(f"  LLM class:       {type(agent.llm).__name__}")
print(f"  Tools:           {list(agent.tools_by_name)}")
print(f"  Memory window:   {agent_config['memory_window']} turns")
print(f"  Max iterations:  {agent_config['max_iterations']} ReAct steps")
print(f"  Safety guard:    {type(agent.guard).__name__}")
print(f"  Feedback memory: {type(agent.feedback).__name__}")
print(f"  Feedback path:   {agent.feedback.history_path}")
print(f"  History loaded:  {len(agent.feedback._buffer)} record(s)")

In [ ]:
# Run a live query. verbose=True prints every ReAct step:
# [Step N]  Thought / Action / Action Input / Observation
# This is the proof that our custom loop is running — not a black-box API.

query = (
    "Write a Python function that checks whether a string is a palindrome, "
    "ignoring case and spaces. Use the linter to verify it, "
    "then run it against three test cases."
)
print("USER:", query)
print("=" * 65)

response = agent.run(query)

print("\n" + "=" * 65)
print("SHAPE FINAL ANSWER")
print("=" * 65)
print(response)

In [ ]:
# Prove conversation memory works (InMemoryChatMessageHistory).
# The agent should reference the palindrome function from the previous turn.

follow_up = (
    "Now extend that same function to also count how many palindromes "
    "exist in a list of strings and return a summary dict."
)
print("FOLLOW-UP:", follow_up)
print("=" * 65)

response2 = agent.run(follow_up)

print("\n" + "=" * 65)
print("SHAPE FINAL ANSWER")
print("=" * 65)
print(response2)

n_msgs = len(agent._chat_history.messages)
print(f"\nHistory buffer: {n_msgs} messages stored (window={agent_config['memory_window']} turns * 2)")

## What This Notebook Demonstrated

| Concept | File | Evidence |
|---|---|---|
| Three-tier hardware | `hardware/pico2/main.py`, `zero2w/`, `pi5/` | UART CSV protocol live-reproduced |
| ReAct parser (no tool API) | `shape/agent.py:_parse_llm_output` | All 3 branches tested with real regexes |
| Tool registry + error recovery | `shape/tools/__init__.py` | Unknown tool → recoverable Observation |
| Agent initialization | `shape/agent.py:build_agent` | Real objects, real paths, logged |
| Full ReAct trace | `shape/agent.py:run` with `verbose=True` | Thought/Action/Obs printed live |
| Conversation memory | `langchain_core.InMemoryChatMessageHistory` | Follow-up uses prior context |

---
▶ Continue to `02_Why_We_Built_SHAPE.ipynb`
